# YOLOv11n — egg detection live demo

This notebook runs YOLOv11n on the Kria KV260's B4096 DPU for live egg
detection, with three input modes:

- **USB camera** — frames from `/dev/video0`. Useful for live deployment scenarios.
- **Video file** — frames from an .mp4 / .avi / .mov file on disk. Useful for
  reproducible demos against pre-recorded footage.
- **Image folder** — iterates one image at a time from a directory.
  Useful for offline testing without a camera.

## Prerequisites

1. xmodel at `/home/ubuntu/xmodels_vai35/yolov11n/yolov11n_kv260.xmodel`
   — produced by `scripts/host/02_compile.sh yolov11 yolov11n ...` on
   the laptop and synced via `scripts/host/03_sync_to_kria.sh`.

2. Running as root (required for FPGA register mmap).
   Use `bash scripts/kria/run_live.sh yolov11n` from the host shell,
   which launches Jupyter with the correct environment.

3. The Kria-side repo synced (`lpr_pipeline` package must include the
   v0.8.0 changes for `family="yolov11"` support).

In [ ]:
# Repo path setup. This notebook lives at notebooks/eggs/05_deploy_visual.ipynb,
# so the repo root is two levels up. The lpr_pipeline package is at the repo root.
import os, sys
from pathlib import Path

# Prefer the env var set by run_live.sh; fall back to walking up from notebook location.
_repo_env = os.environ.get("REPO_ROOT")
if _repo_env and Path(_repo_env).is_dir():
    REPO_ROOT = Path(_repo_env)
else:
    # __file__ isn't set in notebooks; walk from cwd. When launched via run_live.sh
    # the cwd is REPO_ROOT/notebooks, so walking up once works. If launched manually
    # from notebooks/eggs/, walking up twice works. Try both.
    for candidate in (Path.cwd().parent, Path.cwd().parent.parent, Path.cwd()):
        if (candidate / "lpr_pipeline").is_dir():
            REPO_ROOT = candidate
            break
    else:
        raise RuntimeError(
            "Could not locate repo root. Set REPO_ROOT env var, or launch via "
            "scripts/kria/run_live.sh which sets it automatically."
        )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"lpr_pipeline located: {(REPO_ROOT / 'lpr_pipeline').is_dir()}")

In [ ]:
import time
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# PYNQ-DPU
from pynq_dpu import DpuOverlay

# Our pipeline
from lpr_pipeline.shared.models   import get_spec
from lpr_pipeline.deploy.runner   import ModelRunner
from lpr_pipeline.deploy.preprocess import unletterbox

print("Imports OK")

## Input source

Pick one of the three input modes. The selected source is used by the main
inference loop below. Re-run this cell if you change your mind.

In [ ]:
# Three input modes. Pick one by setting INPUT_MODE.
# The cell below this one will display interactive widgets you can use to
# change INPUT_MODE without editing code.

INPUT_MODE = "camera"       # "camera", "video", or "image_folder"

# Per-mode configuration. Edit the values for your setup:
CAMERA_DEVICE_INDEX = 0     # /dev/video0 (the typical first USB camera)
VIDEO_FILE_PATH     = "/home/ubuntu/yolov11n_test/sample_video.mp4"   # your video
IMAGE_FOLDER_PATH   = "/home/ubuntu/yolov11n_test"                     # folder of .jpg/.png

# Detection thresholds (egg-specific defaults from the validation pass).
# Eggs trained model produces real detections at 0.90+ and dataset-bias
# false positives below 0.85 — see docs/YOLOV11.md → Known limitations.
CONF_THRESHOLD = 0.85
IOU_THRESHOLD  = 0.45

# Optional cosmetics
SHOW_LABELS    = True
SHOW_CONFIDENCE = True
SHOW_FPS_OVERLAY = True
CLASS_NAMES    = {0: "egg"}

print(f"Selected input mode: {INPUT_MODE}")
print(f"Conf threshold:      {CONF_THRESHOLD}")
print(f"IoU threshold:       {IOU_THRESHOLD}")

In [ ]:
# Interactive widget to change INPUT_MODE without re-editing the cell above.
# After clicking, the dispatch below uses the widget's value.
mode_selector = widgets.RadioButtons(
    options=[("USB camera (/dev/video0)", "camera"),
             ("Video file",                "video"),
             ("Image folder",              "image_folder")],
    value=INPUT_MODE,
    description="Input:",
    style={"description_width": "initial"},
)

camera_idx_w = widgets.IntText(
    value=CAMERA_DEVICE_INDEX, description="Camera idx:",
    style={"description_width": "initial"},
)
video_path_w = widgets.Text(
    value=VIDEO_FILE_PATH, description="Video path:",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "initial"},
)
folder_path_w = widgets.Text(
    value=IMAGE_FOLDER_PATH, description="Folder path:",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "initial"},
)

display(widgets.VBox([mode_selector, camera_idx_w, video_path_w, folder_path_w]))

print("\nAfter setting widgets, run the cell below to apply them to the variables.")

In [ ]:
# Apply the widget values back to the module-level variables.
# Re-run this cell after changing the widgets above.
INPUT_MODE          = mode_selector.value
CAMERA_DEVICE_INDEX = camera_idx_w.value
VIDEO_FILE_PATH     = video_path_w.value
IMAGE_FOLDER_PATH   = folder_path_w.value

# Validate the source for the chosen mode
if INPUT_MODE == "camera":
    cam_path = f"/dev/video{CAMERA_DEVICE_INDEX}"
    if not Path(cam_path).exists():
        print(f"⚠ {cam_path} does not exist. Plug in your USB camera and re-run.")
    else:
        print(f"✓ Camera device {cam_path} present")
elif INPUT_MODE == "video":
    p = Path(VIDEO_FILE_PATH)
    if not p.is_file():
        print(f"⚠ Video file not found: {p}")
    else:
        print(f"✓ Video file found: {p}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")
elif INPUT_MODE == "image_folder":
    p = Path(IMAGE_FOLDER_PATH)
    if not p.is_dir():
        print(f"⚠ Image folder not found: {p}")
    else:
        n_imgs = sum(1 for f in p.iterdir()
                     if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"))
        print(f"✓ Image folder found: {p}  ({n_imgs} images)")
else:
    print(f"⚠ Unknown INPUT_MODE: {INPUT_MODE}")

## Frame iterator

Each input mode is wrapped in a generator that yields BGR frames. The main
inference loop just consumes from `frame_iter()` without caring where the
frames came from.

In [ ]:
def make_camera_iter(device_index: int = 0):
    """Yield frames from a USB camera via the LPR pipeline's ThreadedCamera.

    ThreadedCamera runs a background thread that continuously drains the
    camera's frame buffer, ensuring read_latest() always returns the most
    recent frame (not a stale buffered one). Without this, VideoCapture.read()
    on a fast camera but slow consumer (us) returns frames from the buffer
    that are 50-100ms old, capping our effective fps regardless of how fast
    we actually process. The thread fixes that.
    """
    from lpr_pipeline.deploy.camera import ThreadedCamera

    cam = ThreadedCamera(device_index=device_index)
    try:
        while True:
            frame = cam.read_latest()
            if frame is None:
                # Camera not ready yet or transient hiccup; back off briefly
                time.sleep(0.005)
                continue
            yield frame
    finally:
        cam.release()


def make_video_iter(path: str, loop: bool = True):
    """Yield frames from a video file, optionally looping when EOF."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {path}")
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                if loop:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                    continue
                else:
                    break
            yield frame
    finally:
        cap.release()


def make_image_folder_iter(folder: str, loop: bool = True,
                           delay_ms: int = 500):
    """Yield frames from a directory of images, with optional looping
    and inter-image pause (so the user can see each result)."""
    folder = Path(folder)
    paths = sorted(
        p for p in folder.iterdir()
        if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
    )
    if not paths:
        raise RuntimeError(f"No images in {folder}")
    while True:
        for p in paths:
            img = cv2.imread(str(p))
            if img is None:
                print(f"  ⚠ could not read {p.name}; skipping")
                continue
            yield img
            time.sleep(delay_ms / 1000.0)
        if not loop:
            break


def frame_iter():
    """Dispatch to the right iterator based on INPUT_MODE."""
    if INPUT_MODE == "camera":
        return make_camera_iter(CAMERA_DEVICE_INDEX)
    elif INPUT_MODE == "video":
        return make_video_iter(VIDEO_FILE_PATH)
    elif INPUT_MODE == "image_folder":
        return make_image_folder_iter(IMAGE_FOLDER_PATH)
    else:
        raise ValueError(f"Unknown INPUT_MODE: {INPUT_MODE}")


print(f"frame_iter configured for: {INPUT_MODE}")

## DPU initialization

Load the dpu.bit bitstream onto the FPGA, then load the yolov11n xmodel
onto the DPU runner.

In [ ]:
# xmodel path. Prefer the env var (set by run_live.sh), else use the default
# location where 03_sync_to_kria.sh places it.
XMODEL_PATH = os.environ.get(
    "LPR_XMODEL",
    "/home/ubuntu/xmodels_vai35/yolov11n/yolov11n_kv260.xmodel",
)

if not Path(XMODEL_PATH).is_file():
    raise FileNotFoundError(
        f"xmodel not found at {XMODEL_PATH}. From your laptop:\n"
        f"  bash scripts/host/03_sync_to_kria.sh ubuntu@<kria-ip> yolov11n"
    )

print(f"xmodel: {XMODEL_PATH}  ({Path(XMODEL_PATH).stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
# Load the DPU overlay. This programs the FPGA fabric with the DPU bitstream
# (one-time, ~3-5 seconds). After this we can hot-swap xmodels onto the DPU
# via overlay.load_model().
overlay = DpuOverlay("dpu.bit")
print("✓ DPU overlay loaded")

# Construct ModelRunner. ModelRunner uses the spec.family field to dispatch
# to the right decoder. For yolov11n, that's decode_yolov11() which is a thin
# alias for decode_yolov5u() (same DFL math).
spec = get_spec("yolov11n")
print(f"  spec: family={spec.family}, imgsz={spec.imgsz}, "
      f"nc={spec.nc}, reg_max={spec.reg_max}")

runner = ModelRunner(spec, XMODEL_PATH, overlay)
print(f"✓ ModelRunner constructed")
print(f"  input dims:  {runner.input_dims}")
print(f"  output dims: {runner.output_dims}")

In [ ]:
# Warmup the JIT and DPU buffers — first call is always slow.
print("Warmup (3 dummy inferences)...")
warmup_times = runner.warmup(n=3, print_each=True)
print(f"\nSteady-state inference: ~{warmup_times[-1]:.1f} ms")

In [ ]:
def draw_detections_eggs(frame_bgr: np.ndarray, dets, conf_threshold: float = 0.0):
    """Draw detection boxes in green. Class label 'egg' (single-class).

    dets: list of (x1, y1, x2, y2, score, class_idx) tuples in original
    camera-frame coords (ModelRunner.infer returns these already un-letterboxed).
    """
    out = frame_bgr.copy()
    for (x1, y1, x2, y2, conf, cls) in dets:
        if conf < conf_threshold:
            continue
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
        if SHOW_LABELS:
            class_name = CLASS_NAMES.get(int(cls), str(cls))
            label = f"{class_name} {conf:.2f}" if SHOW_CONFIDENCE else class_name
            (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(out, (x1, y1 - th - bl - 4),
                          (x1 + tw + 4, y1), (0, 255, 0), -1)
            cv2.putText(out, label, (x1 + 2, y1 - bl - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return out


def draw_stats_overlay(frame_bgr: np.ndarray, fps: float,
                       inf_ms: float, n_dets: int):
    """Top-left panel with FPS, inference latency, detection count."""
    out = frame_bgr.copy()
    lines = [
        f"FPS:  {fps:5.1f}",
        f"DPU:  {inf_ms:5.1f} ms",
        f"Det:  {n_dets:3d}",
        f"Mode: {INPUT_MODE}",
    ]
    pad = 8
    box_h = pad + len(lines) * 22 + pad
    cv2.rectangle(out, (0, 0), (200, box_h), (0, 0, 0), -1)
    cv2.rectangle(out, (0, 0), (200, box_h), (0, 255, 0), 1)
    for i, line in enumerate(lines):
        cv2.putText(out, line, (pad, pad + 18 + i * 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 1)
    return out


print("Visualization helpers defined.")

## Live inference loop

The cell below runs the main inference loop. It displays each frame inline
in the notebook with bounding boxes. Interactive sliders let you adjust
the confidence and IoU thresholds in real time.

**To stop the loop**: interrupt the kernel (□ button in the toolbar or
`Ctrl+M I`). The loop catches `KeyboardInterrupt` and cleans up.

In [ ]:
# Live-tunable widgets for the inference loop
conf_slider = widgets.FloatSlider(
    value=CONF_THRESHOLD, min=0.0, max=1.0, step=0.05,
    description="conf:", continuous_update=True,
    style={"description_width": "initial"},
)
iou_slider = widgets.FloatSlider(
    value=IOU_THRESHOLD, min=0.0, max=1.0, step=0.05,
    description="iou:", continuous_update=True,
    style={"description_width": "initial"},
)
show_overlay_w = widgets.Checkbox(
    value=SHOW_FPS_OVERLAY, description="FPS overlay",
)
display_w = widgets.Image(format="jpeg")

controls = widgets.HBox([conf_slider, iou_slider, show_overlay_w])
display(widgets.VBox([controls, display_w]))

# Main loop
frames_processed = 0
total_detections = 0
loop_start = time.perf_counter()
last_fps_check = loop_start
fps = 0.0

try:
    for frame in frame_iter():
        # Inference
        dets, timings = runner.infer(
            frame,
            conf=conf_slider.value,
            iou=iou_slider.value,
        )
        total_detections += len(dets)

        # Annotate
        annotated = draw_detections_eggs(frame, dets,
                                         conf_threshold=conf_slider.value)
        if show_overlay_w.value:
            annotated = draw_stats_overlay(annotated, fps,
                                           timings["dpu"], len(dets))

        # Downscale before JPEG to 960x540 — encoding 1920x1080 at 85 quality
        # takes ~45 ms on the Cortex-A53 and was capping us at ~20fps. Half
        # resolution + quality 75 brings this down to ~10ms with no
        # visible difference in a Jupyter cell.
        if annotated.shape[1] > 960:
            display_frame = cv2.resize(annotated, (960, 540),
                                       interpolation=cv2.INTER_AREA)
        else:
            display_frame = annotated
        ok, buf = cv2.imencode(".jpg", display_frame,
                               [cv2.IMWRITE_JPEG_QUALITY, 75])
        if ok:
            display_w.value = buf.tobytes()

        # FPS measurement (smoothed over 1 second)
        frames_processed += 1
        now = time.perf_counter()
        if now - last_fps_check >= 1.0:
            fps = frames_processed / (now - loop_start)
            last_fps_check = now

except KeyboardInterrupt:
    elapsed = time.perf_counter() - loop_start
    print(f"\n─── Stopped ───")
    print(f"  Frames processed : {frames_processed}")
    print(f"  Total detections : {total_detections}")
    print(f"  Elapsed time     : {elapsed:.1f} s")
    if elapsed > 0:
        print(f"  Average FPS      : {frames_processed / elapsed:.1f}")
finally:
    pass

## Cleanup

Run this after stopping the loop to free the camera (if any) and release
any held references. Safe to run multiple times.

In [ ]:
# OpenCV's VideoCapture is released by the generator's `finally` clause
# when the generator is GC'd, but we can force it.
import gc
gc.collect()
print("Cleanup done.")